# Ex.No 10 — Compiler Back-End: TAC to 8086 Assembly Code


## AIM
To write a program using FLEX and BISON to implement the back-end of a compiler which takes three-address code (TAC) as input and generates equivalent 8086 assembly language code.


## ALGORITHM / PROCEDURE
1. Use FLEX to tokenize each TAC line into identifiers and the operators `= + - * / ;`, passing tokens to BISON.
2. In BISON, define a grammar for assignment statements: `id = expr ;`.
3. While reducing `expr`:
   - On the first operand, emit `MOV AX, operand`.
   - On `+`, emit `ADD AX, operand`; on `-`, emit `SUB AX, operand`.
   - On `*`, emit `MUL operand`; on `/`, emit `MOV DX,0` / `MOV BX,operand` / `DIV BX`.
4. When the full statement is reduced, emit `MOV result, AX`.
5. Repeat for every TAC statement in the input and print all generated instructions.
6. End.

**Procedure**
1. Create `backend.l` to tokenize identifiers and operators from TAC lines.
2. Create `backend.y` with a grammar for TAC assignment statements, embedding 8086 `MOV`/`ADD`/`SUB`/`MUL`/`DIV` instruction generation directly in the semantic actions.
3. Compile: `flex backend.l` → `bison -d backend.y` → `gcc lex.yy.c backend.tab.c -o backend -lfl`.
4. Run `./backend`, input TAC statements such as `t1 = a + b` followed by `x = t1`, and view the generated 8086 assembly code.


## PSEUDOCODE / LOGIC
```
GRAMMAR:
    stmt -> ID '=' expr ';'          { EMIT "MOV " + ID + ", AX" }
    expr -> ID                       { EMIT "MOV AX, " + ID }
          | expr '+' ID              { EMIT "ADD AX, " + ID }
          | expr '-' ID              { EMIT "SUB AX, " + ID }
          | expr '*' ID              { EMIT "MUL " + ID }
          | expr '/' ID              { EMIT "MOV DX, 0"
                                        EMIT "MOV BX, " + ID
                                        EMIT "DIV BX" }

BEGIN
    FOR each TAC statement in input:
        CALL yyparse() -> emit instructions per rules above
    PRINT all generated 8086 assembly instructions
END
```


## PROGRAM & OUTPUT
The cells below contain the source program (FLEX/BISON/C) and its executed output.


In [84]:
# ============================================================
# BACK-END OF COMPILER USING FLEX AND BISON
# TAC -> 8086 ASSEMBLY LANGUAGE
# Google Colab - Single Cell
# ============================================================

# Install FLEX, BISON and GCC
!apt-get update -qq
!apt-get install -y flex bison gcc -qq


# ============================================================
# Create backend.l
# ============================================================

with open("backend.l", "w") as f:
    f.write(r'''
%{
#include "backend.tab.h"
#include <string.h>
#include <stdlib.h>
%}

%option noyywrap

%%

[a-zA-Z][a-zA-Z0-9]* {
    yylval.str = strdup(yytext);
    return ID;
}

"="     { return '='; }
"+"     { return '+'; }
"-"     { return '-'; }
"*"     { return '*'; }
"/"     { return '/'; }
";"     { return ';'; }

[ \t\n]+ {
    /* Ignore whitespace */
}

. {
    return yytext[0];
}

%%
''')


# ============================================================
# Create backend.y
# ============================================================

with open("backend.y", "w") as f:
    f.write(r'''
%{
#include <stdio.h>
#include <string.h>
#include <stdlib.h>

int yylex(void);
int yyerror(char *s);
%}

%union {
    char *str;
}

%token <str> ID
%type <str> expr

%left '+' '-'
%left '*' '/'

%%

stmt_list:
      stmt_list stmt
    | stmt
    ;

stmt:
    ID '=' expr ';'
    {
        printf("MOV %s, AX\n", $1);
    }
    ;

expr:
      ID
      {
          printf("MOV AX, %s\n", $1);
          $$ = $1;
      }

    | expr '+' ID
      {
          printf("ADD AX, %s\n", $3);
          $$ = $3;
      }

    | expr '-' ID
      {
          printf("SUB AX, %s\n", $3);
          $$ = $3;
      }

    | expr '*' ID
      {
          printf("MUL %s\n", $3);
          $$ = $3;
      }

    | expr '/' ID
      {
          printf("MOV DX, 0\n");
          printf("MOV BX, %s\n", $3);
          printf("DIV BX\n");
          $$ = $3;
      }
    ;

%%

int main()
{
    printf("Enter TAC statements (end with Ctrl+D):\n");
    yyparse();

    return 0;
}

int yyerror(char *s)
{
    printf("Syntax Error: %s\n", s);
    return 0;
}
''')


# ============================================================
# Remove old generated files
# ============================================================

!rm -f backend.tab.c backend.tab.h lex.yy.c backend


# ============================================================
# Generate BISON and FLEX files
# ============================================================

!bison -d backend.y
!flex backend.l


# ============================================================
# Compile
# ============================================================

!gcc lex.yy.c backend.tab.c -o backend -lfl


# ============================================================
# SAMPLE INPUT
# ============================================================

with open("input.txt", "w") as f:
    f.write("""t1 = a + b;
t2 = t1 - c;
t3 = t2 * d;
t4 = t3 / e;
x = t4;
""")


# ============================================================
# Execute
# ============================================================

import subprocess

result = subprocess.run(
    ["./backend"],
    stdin=open("input.txt", "r"),
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

Enter TAC statements (end with Ctrl+D):
MOV AX, a
ADD AX, b
MOV t1, AX
MOV AX, t1
SUB AX, c
MOV t2, AX
MOV AX, t2
MUL d
MOV t3, AX
MOV AX, t3
MOV DX, 0
MOV BX, e
DIV BX
MOV t4, AX
MOV AX, t4
MOV x, AX



## RESULT
Thus, the back-end of the compiler was successfully implemented using FLEX and BISON to translate three-address code into equivalent 8086 assembly language code.
